# Where Am I? Position in Attention, from Sinusoids to RoPE

The runnable code of [Where Am I? Position in Attention, from Sinusoids to RoPE](https://residual-stream.lvtan71.workers.dev/posts/rope-and-positional-encoding/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rope-and-positional-encoding.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "rope-and-positional-encoding"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Attention can't see order

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rope-and-positional-encoding/#attention-cant-see-order)

In [ ]:
import numpy as np

# 1. Attention without positions can't tell order apart: permute the tokens,
#    and every output is permuted the same way; nothing else changes. With a
#    causal mask that's no longer true: each token sees a different prefix.
rng = np.random.default_rng(0)
T, H = 5, 8
X = rng.normal(size=(T, H))
Wq, Wk, Wv = (rng.normal(size=(H, H)) / np.sqrt(H) for _ in range(3))
def attend(X, causal=False):
    s = (X @ Wq) @ (X @ Wk).T / np.sqrt(H)
    if causal:
        s = np.where(np.tri(T, dtype=bool), s, -np.inf)
    w = np.exp(s - s.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return w @ (X @ Wv)
perm = rng.permutation(T)
print(f"attention(permuted input) == permuted attention(input): "
      f"{np.allclose(attend(X[perm]), attend(X)[perm])}")
print(f"the same, with a causal mask: "
      f"{np.allclose(attend(X[perm], causal=True), attend(X, causal=True)[perm])}")

# 2. RoPE: rotate each pair of dimensions (2i, 2i+1) by angle pos * theta_i.
def rope(x, pos, base=10_000.0):
    half = x.shape[-1] // 2
    theta = base ** (-np.arange(half) / half)             # one frequency per pair
    ang = pos * theta
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = np.empty_like(x)
    out[..., 0::2] = x1 * np.cos(ang) - x2 * np.sin(ang)
    out[..., 1::2] = x1 * np.sin(ang) + x2 * np.cos(ang)
    return out

q, k = rng.normal(size=H), rng.normal(size=H)
for m, n in [(3, 5), (10, 12), (100, 102), (3, 10)]:
    print(f"query at {m:3d}, key at {n:3d} (distance {n - m:2d}): score {rope(q, m) @ rope(k, n):+.4f}")
print(f"rotation keeps length: |q| = {np.linalg.norm(q):.4f}, |rope(q, 57)| = {np.linalg.norm(rope(q, 57)):.4f}")

The post printed:

```text
attention(permuted input) == permuted attention(input): True
the same, with a causal mask: False
query at   3, key at   5 (distance  2): score -6.3341
query at  10, key at  12 (distance  2): score -6.3341
query at 100, key at 102 (distance  2): score -6.3341
query at   3, key at  10 (distance  7): score -5.9618
rotation keeps length: |q| = 4.4482, |rope(q, 57)| = 4.4482
```

## A test: two tasks, four times the length

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rope-and-positional-encoding/#a-test-two-tasks-four-times-the-length)

In [ ]:
%%writefile step2.py
import math
import sys
import torch

# A tiny causal Transformer, trained on sequences of 32 and tested on 32, 64
# and 128, on one of two tasks:
#   lag:  output, at every position, the token 3 positions earlier. It needs
#         relative position, always at the same short distance.
#   seen: say whether the current token appeared anywhere earlier. It needs
#         to look back over every distance, including ones longer than any
#         training sequence.
# Kinds: none, sinusoid, learned, rope (or rope:<base>), alibi:<slope>.
torch.set_num_threads(2)
TASK = sys.argv[3] if len(sys.argv) > 3 else "lag"
V = 16 if TASK == "lag" else 96
D, LAG, TRAIN_T = 64, 3, 32
SCALE = 1.0          # multiplies every attention score; 1 in training

def targets(x):
    """(the positions scored, their correct outputs)"""
    if TASK == "lag":
        return slice(LAG, None), x[:, :-LAG]
    earlier = (x[:, :, None] == x[:, None, :]).tril(-1).any(-1)   # token t appeared before t
    return slice(1, None), earlier[:, 1:].long()

def sinusoids(T, D):
    pos, i = torch.arange(T)[:, None], torch.arange(0, D, 2)
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(pos / 10_000 ** (i / D)); pe[:, 1::2] = torch.cos(pos / 10_000 ** (i / D))
    return pe

def rope(x, base=10_000.0):                                   # x: (batch, T, D)
    T, half = x.shape[1], x.shape[2] // 2
    ang = torch.arange(T)[:, None] * base ** (-torch.arange(half) / half)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * ang.cos() - x2 * ang.sin(), x1 * ang.sin() + x2 * ang.cos()], -1).flatten(2)

class Layer(torch.nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.kind, _, arg = kind.partition(":")
        self.arg = float(arg) if arg else 10_000.0
        self.qkv, self.o = torch.nn.Linear(D, 3 * D), torch.nn.Linear(D, D)
        self.mlp = torch.nn.Sequential(torch.nn.Linear(D, 4 * D), torch.nn.GELU(), torch.nn.Linear(4 * D, D))
        self.n1, self.n2 = torch.nn.LayerNorm(D), torch.nn.LayerNorm(D)
    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.qkv(self.n1(x)).chunk(3, -1)
        if self.kind == "rope":
            q, k = rope(q, self.arg), rope(k, self.arg)
        s = SCALE * q @ k.transpose(1, 2) / math.sqrt(D)
        if self.kind == "alibi":                                # a penalty growing with distance
            dist = torch.arange(T)[:, None] - torch.arange(T)[None]
            s = s - self.arg * dist.clamp(min=0)
        s = s.masked_fill(torch.ones(T, T).triu(1).bool(), float("-inf"))
        x = x + self.o(torch.softmax(s, -1) @ v)
        return x + self.mlp(self.n2(x))

class Model(torch.nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.kind = kind.partition(":")[0]
        self.emb, self.out = torch.nn.Embedding(V, D), torch.nn.Linear(D, V if TASK == "lag" else 2)
        self.pos = torch.nn.Embedding(128, D) if kind == "learned" else None
        self.layers = torch.nn.ModuleList([Layer(kind) for _ in range(2)])
    def forward(self, x):
        h = self.emb(x)
        T = x.shape[1]
        if self.kind == "sinusoid":
            h = h + sinusoids(T, D)
        if self.kind == "learned":
            h = h + self.pos(torch.arange(T))                   # rows past 32 are never trained
        for layer in self.layers:
            h = layer(h)
        return self.out(h)

def accuracy(model, T, g, past=0):
    """Accuracy on 256 fresh sequences of length T, counting only
    positions from `past` on."""
    x = torch.randint(0, V, (256, T), generator=g)
    where, y = targets(x)
    with torch.no_grad():
        pred = model(x)[:, where].argmax(-1)
    first = max(0, past - where.start)
    return (pred == y)[:, first:].float().mean().item()

kind, seed = sys.argv[1], int(sys.argv[2])
torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
model = Model(kind)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(2000):
    x = torch.randint(0, V, (64, TRAIN_T), generator=g)
    where, y = targets(x)
    logits = model(x)[:, where]
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
test = lambda: torch.Generator().manual_seed(100 + seed)       # the same test sequences each time
g = test()
line = " / ".join(f"{accuracy(model, T, g):.3f}" for T in (32, 64, 128))
past = accuracy(model, 128, test(), past=32)
SCALE = math.log(128) / math.log(32)                               # log-n scaling, at test time only
scaled = accuracy(model, 128, test())
print(f"{TASK:4s} {kind:12s} seed {seed}: length 32 / 64 / 128: {line}; "
      f"at 128, positions past 32: {past:.3f}; scores x log128/log32: {scaled:.3f}")

In [ ]:
# The post ran the script once per task, position scheme and seed:
for task in ("lag", "seen"):
    for kind in ("none", "sinusoid", "learned", "rope", "rope:1000000", "alibi:0.5", "alibi:0.0625"):
        for seed in (0, 1):
            %run step2.py {kind} {seed} {task}

The post printed:

```text
lag  none         seed 0: length 32 / 64 / 128: 0.522 / 0.339 / 0.240; at 128, positions past 32: 0.158; scores x log128/log32: 0.239
lag  none         seed 1: length 32 / 64 / 128: 0.526 / 0.341 / 0.243; at 128, positions past 32: 0.163; scores x log128/log32: 0.243
lag  sinusoid     seed 0: length 32 / 64 / 128: 1.000 / 0.539 / 0.315; at 128, positions past 32: 0.113; scores x log128/log32: 0.317
lag  sinusoid     seed 1: length 32 / 64 / 128: 1.000 / 0.531 / 0.307; at 128, positions past 32: 0.095; scores x log128/log32: 0.301
lag  learned      seed 0: length 32 / 64 / 128: 1.000 / 0.520 / 0.292; at 128, positions past 32: 0.078; scores x log128/log32: 0.291
lag  learned      seed 1: length 32 / 64 / 128: 1.000 / 0.516 / 0.293; at 128, positions past 32: 0.079; scores x log128/log32: 0.291
lag  rope         seed 0: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  rope         seed 1: length 32 / 64 / 128: 1.000 / 1.000 / 0.999; at 128, positions past 32: 0.999; scores x log128/log32: 1.000
lag  rope:1000000 seed 0: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  rope:1000000 seed 1: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  alibi:0.5    seed 0: length 32 / 64 / 128: 0.940 / 0.939 / 0.939; at 128, positions past 32: 0.938; scores x log128/log32: 0.878
lag  alibi:0.5    seed 1: length 32 / 64 / 128: 0.942 / 0.936 / 0.936; at 128, positions past 32: 0.938; scores x log128/log32: 0.867
lag  alibi:0.0625 seed 0: length 32 / 64 / 128: 0.698 / 0.634 / 0.605; at 128, positions past 32: 0.587; scores x log128/log32: 0.645
lag  alibi:0.0625 seed 1: length 32 / 64 / 128: 0.699 / 0.636 / 0.602; at 128, positions past 32: 0.582; scores x log128/log32: 0.643
seen none         seed 0: length 32 / 64 / 128: 0.997 / 0.959 / 0.808; at 128, positions past 32: 0.747; scores x log128/log32: 0.960
seen none         seed 1: length 32 / 64 / 128: 0.997 / 0.963 / 0.813; at 128, positions past 32: 0.754; scores x log128/log32: 0.960
seen sinusoid     seed 0: length 32 / 64 / 128: 1.000 / 0.901 / 0.725; at 128, positions past 32: 0.638; scores x log128/log32: 0.715
seen sinusoid     seed 1: length 32 / 64 / 128: 1.000 / 0.892 / 0.723; at 128, positions past 32: 0.633; scores x log128/log32: 0.715
seen learned      seed 0: length 32 / 64 / 128: 0.999 / 0.760 / 0.638; at 128, positions past 32: 0.524; scores x log128/log32: 0.641
seen learned      seed 1: length 32 / 64 / 128: 0.999 / 0.727 / 0.617; at 128, positions past 32: 0.498; scores x log128/log32: 0.627
seen rope         seed 0: length 32 / 64 / 128: 0.999 / 0.936 / 0.820; at 128, positions past 32: 0.762; scores x log128/log32: 0.844
seen rope         seed 1: length 32 / 64 / 128: 1.000 / 0.942 / 0.813; at 128, positions past 32: 0.753; scores x log128/log32: 0.845
seen rope:1000000 seed 0: length 32 / 64 / 128: 0.999 / 0.968 / 0.870; at 128, positions past 32: 0.827; scores x log128/log32: 0.892
seen rope:1000000 seed 1: length 32 / 64 / 128: 0.999 / 0.967 / 0.837; at 128, positions past 32: 0.785; scores x log128/log32: 0.877
seen alibi:0.5    seed 0: length 32 / 64 / 128: 0.933 / 0.877 / 0.752; at 128, positions past 32: 0.689; scores x log128/log32: 0.785
seen alibi:0.5    seed 1: length 32 / 64 / 128: 0.938 / 0.881 / 0.755; at 128, positions past 32: 0.698; scores x log128/log32: 0.791
seen alibi:0.0625 seed 0: length 32 / 64 / 128: 0.991 / 0.942 / 0.839; at 128, positions past 32: 0.790; scores x log128/log32: 0.652
seen alibi:0.0625 seed 1: length 32 / 64 / 128: 0.993 / 0.940 / 0.831; at 128, positions past 32: 0.780; scores x log128/log32: 0.663
```

## The frequencies, and longer contexts

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rope-and-positional-encoding/#the-frequencies-and-longer-contexts)

In [ ]:
import numpy as np

# Each pair of dimensions rotates at its own speed; its wavelength is how many
# positions a full turn takes. Head dimension 128 (64 pairs), for two bases:
# 10,000 (the original) and 500,000 (Llama 3).
H = 128
i = np.arange(H // 2)
print(f"{'pair':>5s} " + "".join(f"{f'base {b:,}':>18s}" for b in (10_000, 500_000)))
for p in (0, 8, 16, 32, 48, 63):
    print(f"{p:5d} " + "".join(f"{2 * np.pi * b ** (p / (H // 2)):>18,.1f}" for b in (10_000, 500_000)))
for b in (10_000, 500_000):
    wl = 2 * np.pi * b ** (i / (H // 2))
    print(f"base {b:,}: {np.sum(wl > 8192):2d} of 64 pairs never finish one turn within 8,192 positions, "
          f"{np.sum(wl > 131_072):2d} within 131,072")

The post printed:

```text
 pair        base 10,000      base 500,000
    0                6.3               6.3
    8               19.9              32.4
   16               62.8             167.1
   32              628.3           4,442.9
   48            6,283.2         118,142.8
   63           54,410.1       2,559,195.5
base 10,000: 14 of 64 pairs never finish one turn within 8,192 positions,  0 within 131,072
base 500,000: 29 of 64 pairs never finish one turn within 8,192 positions, 15 within 131,072
```